# Health IoT Lakehouse — Feature Exploration
Explore the ML feature table `mart_patient_daily_features` for anomaly prediction.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from trino.dbapi import connect
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
sns.set_theme(style="whitegrid")

In [ ]:
conn = connect(
    host="localhost",
    port=8080,
    user="admin",
    catalog="iceberg",
    schema="marts"
)
df = pd.read_sql(
    "SELECT * FROM mart_patient_daily_features ORDER BY report_date DESC LIMIT 10000",
    conn
)
print(f"Shape: {df.shape}")
df.head()

## Dataset Overview

In [ ]:
print(df.dtypes)
print(df.describe())
print(f"\nAnomaly rate: {df['anomaly_next_24h'].mean():.2%}")
print(f"Date range: {df['report_date'].min()} to {df['report_date'].max()}")
print(f"Unique patients: {df['patient_hk'].nunique()}")

## Target Variable Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['anomaly_next_24h'].value_counts().plot(kind='bar', ax=axes[0], title='Anomaly Label Distribution')
df.groupby('risk_tier')['anomaly_next_24h'].mean().sort_values(ascending=False).plot(
    kind='bar', ax=axes[1], title='Anomaly Rate by Risk Tier'
)
plt.tight_layout()
plt.show()

## Vital Signs Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col, label in zip(axes, ['hr_mean', 'spo2_mean', 'steps_total'], ['HR Mean', 'SpO2 Mean', 'Steps Total']):
    df.boxplot(column=col, by='anomaly_next_24h', ax=ax)
    ax.set_title(label)
plt.suptitle('Vital Signs vs Anomaly Label')
plt.tight_layout()
plt.show()

## Correlation Matrix

In [ ]:
numeric_cols = ['hr_mean', 'hr_std', 'spo2_mean', 'spo2_min', 'steps_total',
                'skin_temp_mean', 'composite_anomaly_score', 'anomaly_events_30d',
                'hr_mean_7d_avg', 'hr_mean_delta_1d']
corr = df[numeric_cols + ['anomaly_next_24h']].corr()
plt.figure(figsize=(12, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')
plt.show()

## Simple Baseline Model

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import LabelEncoder

feature_cols = ['hr_mean', 'hr_std', 'hr_min', 'hr_max', 'spo2_mean', 'spo2_min',
                'steps_total', 'skin_temp_mean', 'composite_anomaly_score',
                'anomaly_events_30d', 'hr_mean_7d_avg', 'hr_mean_delta_1d',
                'age', 'bmi', 'condition_count',
                'has_diabetes', 'has_hypertension', 'has_cardiac_condition']

df_model = df[feature_cols + ['anomaly_next_24h']].dropna()
X = df_model[feature_cols]
y = df_model['anomaly_next_24h'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))
print(f"ROC-AUC: {roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]):.3f}")

In [ ]:
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=True)
plt.figure(figsize=(10, 8))
importances.tail(15).plot(kind='barh')
plt.title('Top 15 Feature Importances (Random Forest)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## Next Steps
- Add more historical data (backfill)
- Tune model hyperparameters
- Add LSTM for temporal patterns
- Use Iceberg time travel to pin training snapshot